# First prototype for HyperBubble Resolution-Oriented GNN

In [1]:
from pathlib import Path
from typing import Any, Dict, List, Optional
import json
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

In [2]:
NUC2ID = { 'A':1, 'C':2, 'G':3, 'T':4, 'N':5 }
def seq_to_tokens(seq: str) -> torch.LongTensor:
    return torch.tensor([NUC2ID.get(c, 5) for c in seq], dtype=torch.long)

def _safe_get(d: Dict[str, Any], key: str, default):
    v = d.get(key, default)
    return default if v is None else v

ID2NUC = {v:k for k,v in NUC2ID.items()}

def tokens_to_seq(tokens: torch.LongTensor) -> str:
    return "".join(ID2NUC.get(int(t), "N") for t in tokens.tolist())

In [3]:
import json
from pathlib import Path
from typing import List, Dict, Any, Optional
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data

class HyperbubbleDataset(Dataset):
    """
    Minimal GNN-ready dataset from your JSONL:
      - Node features:
          seq_tokens : [N, K] Long
          x_cov      : [N, 1] Float (KC coverage; 0 if unknown)
      - Graph:
          edge_index : [2, E] Long
          edge_attr  : [E, 5] Float (len_nodes, len_bp, cov_min, cov_mean, on_ref)
          start_idx, end_idx : Long
      - Labels:
          y_edge_mask    : [E] Long (1 on labeled path edges, else 0)
          label_path_idx : Long (-1 if none)
    """
    def __init__(self, files: List[str]):
        self.files = [Path(p) for p in files]
        self.records: List[Dict[str, Any]] = []
        for fp in self.files:
            with fp.open("r") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue

    def __len__(self) -> int:
        return len(self.records)

    def _build_graph(self, rec: Dict[str, Any]) -> Data:
        # --- nodes (map seq -> idx), carry best-known coverage ---
        node_seqs: Dict[str, int] = {}
        node_covs: List[float] = []

        def ensure_node(seq: str, cov: Optional[float] = None) -> int:
            if seq not in node_seqs:
                node_seqs[seq] = len(node_seqs)
                node_covs.append(float(cov) if cov is not None else 0.0)
            else:
                i = node_seqs[seq]
                if cov is not None and node_covs[i] == 0.0:
                    node_covs[i] = float(cov)
            return node_seqs[seq]

        # endpoints
        start_seq = rec["start_seq"]
        end_seq   = rec["end_seq"]

        # core nodes
        for n in _safe_get(rec, "nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        # 1-hop context
        for n in _safe_get(rec, "upstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))
        for n in _safe_get(rec, "downstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        # ---- Strategy-A selected interior nodes (if present) ----
        for n in _safe_get(rec, "inside_nodes", []):
            if isinstance(n, dict) and "seq" in n:
                ensure_node(n["seq"], n.get("cov", 0))

        # make sure endpoints are present
        ensure_node(start_seq)
        ensure_node(end_seq)

        # --- edges + attributes ---
        edge_src, edge_dst, edge_attr = [], [], []
        for e in _safe_get(rec, "edges", []):
            u = ensure_node(e["source_seq"])
            v = ensure_node(e["target_seq"])
            edge_src.append(u)
            edge_dst.append(v)
            edge_attr.append([
                float(e.get("len_nodes", 0)),
                float(e.get("len_bp", 0)),
                float(e.get("cov_min", 0)),
                float(e.get("cov_mean", 0.0)),
                1.0 if e.get("on_ref", False) else 0.0
            ])

        # --- node features ---
        node_order = [None] * len(node_seqs)
        for s, idx in node_seqs.items():
            node_order[idx] = s

        # keep tokens; embedding happens in GNN
        seq_tokens = torch.stack([seq_to_tokens(s) for s in node_order], dim=0)  # [N, K] Long
        x_cov = torch.tensor(node_covs, dtype=torch.float32).unsqueeze(1)        # [N, 1] Float

        start_idx = torch.tensor(node_seqs[start_seq], dtype=torch.long)
        end_idx   = torch.tensor(node_seqs[end_seq], dtype=torch.long)

        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr  = (torch.tensor(edge_attr, dtype=torch.float32)
                      if edge_attr else torch.zeros((0,5), dtype=torch.float32))

        # --- labels from label_path ---
        num_edges = edge_index.size(1)
        y_edge_mask = torch.zeros(num_edges, dtype=torch.long)
        label_path_idx = -1
        paths_list = _safe_get(rec, "paths", [])
        lp = rec.get("label_path")
        if lp is not None and 0 <= lp < len(paths_list) and num_edges > 0:
            label_path_idx = int(lp)
            y_edge_mask[torch.tensor(paths_list[label_path_idx], dtype=torch.long)] = 1

        data = Data(
            seq_tokens=seq_tokens,                 # [N, K] Long (to be embedded in model)
            x_cov=x_cov,                           # [N, 1] Float
            edge_index=edge_index,                 # [2, E] Long
            edge_attr=edge_attr,                   # [E, 5] Float
            start_idx=start_idx,                   # Long
            end_idx=end_idx,                       # Long
            num_nodes=seq_tokens.size(0),
            y_edge_mask=y_edge_mask,               # [E] Long
            label_path_idx=torch.tensor(label_path_idx, dtype=torch.long),
        )
        if "bubble_id" in rec:
            data.bubble_id = torch.tensor(rec["bubble_id"], dtype=torch.long)
        if "k" in rec:
            data.k = torch.tensor(rec["k"], dtype=torch.long)
        return data

    def __getitem__(self, idx: int) -> Data:
        return self._build_graph(self.records[idx])


In [4]:
from pathlib import Path
from torch.utils.data import Subset
from torch_geometric.loader import DataLoader
import torch

# Always load BOTH
train_paths = [
    "lower_cov_data/ecoli_dataset_cov20_full_upgraded_ratio_lt_35.jsonl",
    "lower_cov_data/salmonella_dataset_cov20_full_upgraded_ratio_lt_35.jsonl",
]
test_paths = [
    "lower_cov_data/klebsiella_dataset_cov20_full_upgraded_ratio_lt_35.jsonl",
]

# Sanity check files exist
for p in train_paths + test_paths:
    if not Path(p).is_file():
        raise FileNotFoundError(f"Missing: {p}")

# Build combined datasets
train_full = HyperbubbleDataset(train_paths)
test_full  = HyperbubbleDataset(test_paths)

def _check_k(ds):
    # grab first two samples and verify K (length of a seq_tokens row)
    if len(ds) == 0: 
        return
    s0 = ds[0].seq_tokens.size(1)
    for i in (len(ds)//2, len(ds)-1):
        s = ds[i].seq_tokens.size(1)
        assert s == s0, f"Inconsistent k across samples: {s0} vs {s}"
_check_k(train_full); _check_k(test_full)

# Keep only labeled samples
def labeled_subset(ds):
    labeled_indices = []
    for i in range(len(ds)):
        d = ds[i]
        lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
        if lp >= 0:
            labeled_indices.append(i)
    return Subset(ds, labeled_indices)

train_dataset = labeled_subset(train_full)
test_dataset  = labeled_subset(test_full)

print(f"Train (labeled): {len(train_dataset)}")
print(f"Test  (labeled): {len(test_dataset)}")

# DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

Train (labeled): 841
Test  (labeled): 283


In [5]:
USE_DIRECTML = True

import torch
device = torch.device('cpu')
if not USE_DIRECTML and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_DIRECTML:
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using DirectML:", device)
    except Exception as e:
        print("DirectML not available, falling back to CPU:", e)
        device = torch.device('cpu')

import torch.nn as nn
import torch.nn.functional as F

class DenseGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, H, edge_index_local, n_nodes: int):
        if n_nodes == 0:
            return H
        A = H.new_zeros((n_nodes, n_nodes))
        if edge_index_local.numel() > 0:
            src, dst = edge_index_local
            one = torch.ones_like(src, dtype=H.dtype)
            A.index_put_((src, dst), one, accumulate=True)

        # add self-loops w/o torch.eye
        idx = torch.arange(n_nodes, device=H.device)
        try:
            A[idx, idx] += 1
        except Exception:
            flat = A.view(-1)
            diag_idx = torch.arange(0, n_nodes*n_nodes, step=n_nodes+1, device=H.device)
            flat.index_add_(0, diag_idx, torch.ones(n_nodes, device=H.device, dtype=H.dtype))
            A = flat.view(n_nodes, n_nodes)

        deg = A.sum(dim=1) + 1e-8
        D_inv_sqrt = deg.pow(-0.5)
        A_norm = (D_inv_sqrt[:, None] * A) * D_inv_sqrt[None, :]
        return A_norm @ self.lin(H)

# ---- Simple GNN with sequence embedding + GCN + edge MLP ----
class HyperbubbleGNN(nn.Module):
    def __init__(
        self,
        vocab_size=5,          # tokens: 0=PAD, A,C,G,T (and optionally N)
        embed_dim=32,          # per-token embedding dim
        gcn_hidden=64,         # node hidden dim (consistent throughout)
        edge_hidden=64,        # hidden in edge MLP
        edge_feat_dim=5,       # [len_nodes, len_bp, cov_min, cov_mean, on_ref]
        use_lstm=False,        # optional: token encoder = mean or BiLSTM
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.use_lstm = use_lstm
        if use_lstm:
            self.lstm = nn.LSTM(embed_dim, gcn_hidden // 2, batch_first=True, bidirectional=False)
            self.node_in = gcn_hidden + 1   # + coverage scalar
        else:
            self.node_in = embed_dim + 1

        self.gcn_hidden = gcn_hidden
        self.gcn1 = DenseGCNLayer(self.node_in, gcn_hidden)
        self.gcn2 = DenseGCNLayer(gcn_hidden, gcn_hidden)

        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden + edge_feat_dim, edge_hidden),
            nn.ReLU(),
            nn.Linear(edge_hidden, 1),
        )

    def encode_nodes(self, seq_tokens, x_cov):
        """
        seq_tokens: [N, K] tokenized k-mer
        x_cov     : [N, 1]
        returns   : [N, node_in]
        """
        E = self.embed(seq_tokens)
        mask = (seq_tokens != 0).float().unsqueeze(-1)
        if self.use_lstm:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1).long()
            mean_init = (E * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()
            H0, _ = self.lstm(E)
            Hseq = (H0 * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()
        else:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1)
            Hseq = (E * mask).sum(dim=1) / lengths.unsqueeze(-1)

        X = torch.cat([Hseq, x_cov], dim=1)
        return X

    def forward(self, data):
        """
        data: PyG batch with fields:
          - seq_tokens [N,K], x_cov [N,1], edge_index [2,E], edge_attr [E,5],
            batch [N] (graph ids), num_nodes, etc.
        returns:
          logits [E]  (edge-wise)
        """
        device = data.seq_tokens.device
        N = data.num_nodes
        E = data.edge_index.size(1)

        X0 = self.encode_nodes(data.seq_tokens, data.x_cov)

        out_node = X0.new_zeros((N, self.gcn_hidden))
        batch_vec = data.batch if hasattr(data, "batch") else torch.zeros(N, dtype=torch.long, device=device)
        num_graphs = int(batch_vec.max().item()) + 1 if N > 0 else 0

        for g in range(num_graphs):
            node_ids = (batch_vec == g).nonzero(as_tuple=False).view(-1)
            n_nodes = int(node_ids.numel())
            if n_nodes == 0:
                continue

            local_map = torch.full((N,), -1, dtype=torch.long, device=device)
            local_map[node_ids] = torch.arange(n_nodes, device=device, dtype=torch.long)

            ei = data.edge_index
            keep = (local_map[ei[0]] >= 0) & (local_map[ei[1]] >= 0)
            keep_idx = keep.nonzero(as_tuple=False).view(-1)
            if keep_idx.numel() == 0:
                H = F.relu(self.gcn1(X0[node_ids], edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                H = F.relu(self.gcn2(H,          edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                out_node[node_ids] = H
                continue

            src_local = local_map[ei[0, keep_idx]]
            dst_local = local_map[ei[1, keep_idx]]
            edge_local = torch.stack([src_local, dst_local], dim=0)

            H = F.relu(self.gcn1(X0[node_ids], edge_local, n_nodes))
            H = F.relu(self.gcn2(H,            edge_local, n_nodes))
            out_node[node_ids] = H

        if E == 0:
            return torch.empty((0,), device=device)

        u, v = data.edge_index
        U = out_node[u]
        V = out_node[v]
        EA = data.edge_attr if hasattr(data, "edge_attr") and data.edge_attr.numel() > 0 \
             else out_node.new_zeros((E, 5))
        feats = torch.cat([U, V, EA], dim=1)
        logits = self.edge_mlp(feats).squeeze(-1)
        return logits


Using DirectML: privateuseone:0


In [6]:
import torch.nn.functional as F

def _num_graphs_in_batch(node_batch: torch.Tensor) -> int:
    return int(node_batch.max().item()) + 1 if node_batch.numel() else 0

def _edge_batch_from_nodes(node_batch: torch.Tensor, edge_src: torch.Tensor) -> torch.Tensor:
    return node_batch[edge_src] if edge_src.numel() else edge_src.new_zeros((0,), dtype=torch.long)

def _collect_decisions_by_labeled_sources(data, g: int, u: torch.Tensor, edge_batch: torch.Tensor):
    """
    Return list of (cand_idx_tensor, gold_pos_tensor) for EVERY decision node
    whose source node appears on the labeled path. This is orientation-agnostic.
    """
    y = data.y_edge_mask
    lab_eidx = (y == 1).nonzero(as_tuple=False).view(-1)
    lab_eidx = lab_eidx[edge_batch[lab_eidx] == g]
    if lab_eidx.numel() == 0:
        return []

    labeled_sources = torch.unique(u[lab_eidx])

    decisions = []
    for s in labeled_sources.tolist():
        mask = (edge_batch == g) & (u == s)
        cand_idx = mask.nonzero(as_tuple=False).view(-1)
        if cand_idx.numel() < 2:
            continue
        y_here = y[cand_idx]
        gold_pos = (y_here == 1).nonzero(as_tuple=False).view(-1)
        if gold_pos.numel() == 1:
            decisions.append((cand_idx, gold_pos.view(1)))
    return decisions

def train_one_epoch_choice_all(model, loader, optim, device):
    model.train()
    total_loss, total_decisions = 0.0, 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        edge_batch = _edge_batch_from_nodes(node_batch, u)
        num_graphs = _num_graphs_in_batch(node_batch)

        batch_loss = 0.0
        dec_used = 0

        for g in range(num_graphs):
            decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
            for cand_idx, gold_pos in decisions:
                ce = F.cross_entropy(logits[cand_idx].unsqueeze(0), gold_pos.view(1))
                batch_loss += ce
                dec_used += 1

        if dec_used == 0:
            continue

        batch_loss = batch_loss / dec_used
        optim.zero_grad()
        batch_loss.backward()
        optim.step()

        total_loss += batch_loss.item()
        total_decisions += dec_used

    return total_loss / max(total_decisions, 1)

def _cpu_state_dict(sd):
    return {k: v.detach().cpu().clone() for k, v in sd.items()}

best_ckpt = {
    "epoch": 0,
    "dec_acc": -1.0,
    "bub_acc": -1.0,
    "state_dict": None,
}

def _is_better(bub_acc, dec_acc, best):
    if bub_acc > best["bub_acc"]:
        return True
    if bub_acc == best["bub_acc"] and dec_acc > best["dec_acc"]:
        return True
    return False

@torch.no_grad()
def eval_choice_all(model, loader, device):
    model.eval()
    total_decisions, correct_decisions = 0, 0
    total_bubbles, correct_bubbles = 0, 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        edge_batch = _edge_batch_from_nodes(node_batch, u)
        num_graphs = _num_graphs_in_batch(node_batch)

        for g in range(num_graphs):
            decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
            if not decisions:
                continue
            total_bubbles += 1

            all_correct = True
            for cand_idx, gold_pos in decisions:
                pred = logits[cand_idx].argmax().item()
                ok = int(pred == gold_pos.item())
                correct_decisions += ok
                total_decisions += 1
                all_correct &= bool(ok)

            correct_bubbles += int(all_correct)

    dec_acc = correct_decisions / max(total_decisions, 1)
    bub_acc = correct_bubbles / max(total_bubbles, 1)
    print(f"[choice-eval] decisions={total_decisions} acc_dec={dec_acc:.3f} | bubbles={total_bubbles} acc_bub={bub_acc:.3f}")
    return dec_acc, bub_acc

import math

class AdamW_NoLerp(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        if lr <= 0.0:
            raise ValueError(f"Invalid lr: {lr}")
        if not 0.0 <= eps:
            raise ValueError(f"Invalid eps: {eps}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta1: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta2: {betas[1]}")
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group["lr"]
            beta1, beta2 = group["betas"]
            eps = group["eps"]
            wd = group["weight_decay"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad

                state = self.state[p]
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state["exp_avg_sq"] = torch.zeros_like(p, memory_format=torch.preserve_format)

                exp_avg = state["exp_avg"]
                exp_avg_sq = state["exp_avg_sq"]

                state["step"] += 1
                t = state["step"]

                # Decoupled weight decay
                if wd != 0.0:
                    p.add_(p, alpha=-lr * wd)

                # EMA updates without lerp
                # exp_avg = beta1*exp_avg + (1-beta1)*grad
                exp_avg.mul_(beta1).add_(grad, alpha=1.0 - beta1)
                # exp_avg_sq = beta2*exp_avg_sq + (1-beta2)*grad*grad
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1.0 - beta2)

                # Bias correction
                bias_c1 = 1.0 - beta1 ** t
                bias_c2 = 1.0 - beta2 ** t
                step_size = lr * math.sqrt(bias_c2) / bias_c1

                denom = exp_avg_sq.sqrt().add_(eps)

                # p = p - step_size * exp_avg / denom
                p.addcdiv_(exp_avg, denom, value=-step_size)

        return loss



In [7]:
# model = HyperbubbleGNN(
#     vocab_size=5, embed_dim=16, gcn_hidden=32, edge_hidden=16, edge_feat_dim=5, use_lstm=False
# )
# model.to(device)

# total_params = sum(p.numel() for p in model.parameters())
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# print(f"Total parameters: {total_params}")
# print(f"Trainable parameters: {trainable_params}")
# print(f"Non-trainable parameters: {total_params - trainable_params}")

# optim = torch.optim.AdamW(model.parameters(), lr=1e-3)

# EPOCHS = 100
# for epoch in range(1, EPOCHS + 1):
#     tr_loss = train_one_epoch_choice_all(model, train_loader, optim, device)
#     print(f"epoch {epoch}: train_choice_loss={tr_loss:.4f}")
#     dec_acc, bub_acc = eval_choice_all(model, test_loader, device)

#     if _is_better(bub_acc, dec_acc, best_ckpt):
#         best_ckpt["epoch"] = epoch
#         best_ckpt["dec_acc"] = dec_acc
#         best_ckpt["bub_acc"] = bub_acc
#         best_ckpt["state_dict"] = _cpu_state_dict(model.state_dict())
#         print(f"[checkpoint] new best @ epoch {epoch}: acc_bub={bub_acc:.4f}, acc_dec={dec_acc:.4f}")

# print(f"\nBest test metrics: epoch={best_ckpt['epoch']} acc_bub={best_ckpt['bub_acc']:.4f} acc_dec={best_ckpt['dec_acc']:.4f}")

In [8]:
# torch.save(best_ckpt["state_dict"], "model2.8k_upgraded_and_augmented_dataset.pth")

In [9]:
# MODEL_PATH = "./model8k_upgraded_and_augmented_dataset.pth"

# model = HyperbubbleGNN(
#     vocab_size=5, embed_dim=64, gcn_hidden=32, edge_hidden=64, edge_feat_dim=5, use_lstm=False
# )
# model.to(device)

# state = torch.load(MODEL_PATH, map_location="cpu")
# model.load_state_dict(state)
# model.eval()

# with torch.no_grad():
#     dec_acc, bub_acc = eval_choice_all(model, test_loader, device)
# print(f"[loaded-eval] acc_bub={bub_acc:.4f} acc_dec={dec_acc:.4f}")

In [10]:
import itertools, copy, time, math

EMBED_DIMS   = [16, 32, 64]
GCN_HIDDENS  = [32, 64, 128]
EDGE_HIDDENS = [16, 32, 64]


EPOCHS_PER_TRIAL = 50
LR = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

def make_model(embed_dim, gcn_hidden, edge_hidden):
    m = HyperbubbleGNN(
        vocab_size=5,
        embed_dim=embed_dim,
        gcn_hidden=gcn_hidden,
        edge_hidden=edge_hidden,
        edge_feat_dim=5,
        use_lstm=False
    )
    return m.to(device)

def is_better(a_bub, a_dec, b_bub, b_dec):
    return (a_bub > b_bub) or (a_bub == b_bub and a_dec > b_dec)

results = []
global_best = {
    "config": None,
    "bub_acc": -1.0,
    "dec_acc": -1.0,
    "state_dict": None,
}

trial_idx = 0
total_trials = len(EMBED_DIMS) * len(GCN_HIDDENS) * len(EDGE_HIDDENS)
start_all = time.time()

for ed, gh, eh in itertools.product(EMBED_DIMS, GCN_HIDDENS, EDGE_HIDDENS):
    trial_idx += 1
    print(f"\n=== Trial {trial_idx}/{total_trials} :: embed_dim={ed}, gcn_hidden={gh}, edge_hidden={eh} ===")

    model = make_model(ed, gh, eh)
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    trial_best = {"epoch": 0, "bub_acc": -1.0, "dec_acc": -1.0, "state_dict": None}
    for epoch in range(1, EPOCHS_PER_TRIAL + 1):
        tr_loss = train_one_epoch_choice_all(model, train_loader, optim, device)

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)

        print(f"epoch {epoch}: train_choice_loss={tr_loss:.4f}")
        dec_acc, bub_acc = eval_choice_all(model, test_loader, device)

        if is_better(bub_acc, dec_acc, trial_best["bub_acc"], trial_best["dec_acc"]):
            trial_best["epoch"] = epoch
            trial_best["bub_acc"] = bub_acc
            trial_best["dec_acc"] = dec_acc
            trial_best["state_dict"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"[trial-best] epoch {epoch}: acc_bub={bub_acc:.4f}, acc_dec={dec_acc:.4f}")

        if bub_acc >= 1.0:
            break

    trial_summary = {
        "embed_dim": ed,
        "gcn_hidden": gh,
        "edge_hidden": eh,
        "best_epoch": trial_best["epoch"],
        "best_bub_acc": trial_best["bub_acc"],
        "best_dec_acc": trial_best["dec_acc"],
    }
    results.append(trial_summary)
    print(f"[trial-done] {trial_summary}")

    if is_better(trial_best["bub_acc"], trial_best["dec_acc"],
                 global_best["bub_acc"], global_best["dec_acc"]):
        global_best["config"] = (ed, gh, eh)
        global_best["bub_acc"] = trial_best["bub_acc"]
        global_best["dec_acc"] = trial_best["dec_acc"]
        global_best["state_dict"] = trial_best["state_dict"]
        print(f"[global-best] config={global_best['config']} "
              f"acc_bub={global_best['bub_acc']:.4f} acc_dec={global_best['dec_acc']:.4f}")

elapsed = time.time() - start_all
print(f"\n=== grid-search complete in {elapsed/60:.1f} min ===")

results_sorted = sorted(results, key=lambda r: (r["best_bub_acc"], r["best_dec_acc"]), reverse=True)
print("\nTop configs:")
for r in results_sorted[:5]:
    print(f"  embed={r['embed_dim']:>3}, gcn={r['gcn_hidden']:>3}, edge={r['edge_hidden']:>3}  "
          f"| bub={r['best_bub_acc']:.4f}  dec={r['best_dec_acc']:.4f}  @epoch {r['best_epoch']}")

if global_best["state_dict"] is not None:
    best_ed, best_gh, best_eh = global_best["config"]
    best_model = make_model(best_ed, best_gh, best_eh)
    best_model.load_state_dict(global_best["state_dict"])
    torch.save({
        "config": {"embed_dim": best_ed, "gcn_hidden": best_gh, "edge_hidden": best_eh},
        "state_dict": global_best["state_dict"],
        "bub_acc": global_best["bub_acc"],
        "dec_acc": global_best["dec_acc"],
    }, "best_hyperbubble_gnn.pt")
    print(f"\nSaved best checkpoint -> best_hyperbubble_gnn.pt  "
          f"(config={global_best['config']}, bub={global_best['bub_acc']:.4f}, dec={global_best['dec_acc']:.4f})")
else:
    print("\nNo valid checkpoints collected.")


=== Trial 1/27 :: embed_dim=16, gcn_hidden=32, edge_hidden=16 ===


C:\Users\Przemek\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torch\optim\adamw.py:529: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


epoch 1: train_choice_loss=0.0124
[choice-eval] decisions=303 acc_dec=0.667 | bubbles=283 acc_bub=0.657
[trial-best] epoch 1: acc_bub=0.6572, acc_dec=0.6667
epoch 2: train_choice_loss=0.0112
[choice-eval] decisions=303 acc_dec=0.620 | bubbles=283 acc_bub=0.618
epoch 3: train_choice_loss=0.0109
[choice-eval] decisions=303 acc_dec=0.634 | bubbles=283 acc_bub=0.629
epoch 4: train_choice_loss=0.0109
[choice-eval] decisions=303 acc_dec=0.620 | bubbles=283 acc_bub=0.615
epoch 5: train_choice_loss=0.0108
[choice-eval] decisions=303 acc_dec=0.640 | bubbles=283 acc_bub=0.636
epoch 6: train_choice_loss=0.0108
[choice-eval] decisions=303 acc_dec=0.640 | bubbles=283 acc_bub=0.636
epoch 7: train_choice_loss=0.0108
[choice-eval] decisions=303 acc_dec=0.624 | bubbles=283 acc_bub=0.618
epoch 8: train_choice_loss=0.0109
[choice-eval] decisions=303 acc_dec=0.634 | bubbles=283 acc_bub=0.629
epoch 9: train_choice_loss=0.0109
[choice-eval] decisions=303 acc_dec=0.653 | bubbles=283 acc_bub=0.647
epoch 10: t